# v2.03 — v2.02 with v2.01's learning rate, batch and epoch choice (validation only)

All values chosen by Siwat (`configs/v2_03.json`; record: `claude/log-entry-2026-10-25-v2-02-decisions.md`, section v2.03). Same as v2.02 except:
- learning rate **constant 1e-4** (v2.01), not cosine;
- batch **256** (v2.02 peaked at 5 GB at batch 128);
- kept epoch = **best mean macro-F1** of the EMA weights (v2.01's rule), with **early stopping** 5 epochs after it, at most **30 epochs**.

Kept from v2.02: patches 2 / 5 / 21, SMA / EMA 9 / 21 / 60, EMA weights (decay 0.99, BatchNorm recomputed), per-epoch diagnostics, 3-seed ensemble alongside.

Why: in v2.02 seed 1 the cross-entropy-gated rule kept epoch 2 (hold-out F1 0.350) although hold-out F1 reached 0.379 at epoch 7.

**Runtime: A100.** The three seeds train **in parallel** on the one GPU (about 10 GB each at batch 256), then `scripts/merge_v2xx.py` combines them into the usual result; each seed trains exactly as it would alone. Results in `results/v2xx/A100/` (per-seed parts and logs in `v2_03_parts/`). Test block stays sealed.

In [ ]:
!nvidia-smi -L
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f'GPU: {name}, {mem:.0f} GB')
GPU_TAG = 'A100' if 'A100' in name else 'L4' if 'L4' in name else None
assert GPU_TAG, 'stop: switch the runtime to an A100 or L4 GPU (Runtime -> Change runtime type)'
print('results folder tag:', GPU_TAG)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT = f'{SENIOR}/prices', f'{SENIOR}/results/v2xx/{GPU_TAG}'   # one folder per GPU
RUN_WALK_FORWARD = False   # Siwat, 2026-10-19: official window only
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, json, subprocess, sys
from pathlib import Path
for f in ['scripts/train_v2xx.py', 'src/proposal_model.py', 'src/encoders.py', 'src/features_proposal.py', 'src/train_utils.py',
          'src/stopping.py', 'src/run_config.py', 'configs/v2_03.json', 'tests/test_v2xx.py', 'tests/test_v2xx_training.py',
          'tests/test_v2xx_pipeline.py', 'scripts/plot_v2xx_training.py', 'scripts/merge_v2xx.py']:
    assert os.path.exists(f), f'stop: push the v2.03 config first ({f} missing)'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# 3. Config must be complete
sys.path.insert(0, 'src')
import run_config as RC
cfg = RC.load('configs/v2_03.json')
print('v2.03 ready:', json.dumps({k: cfg[k] for k in ('model', 'loss', 'training', 'report')}, indent=1))

In [ ]:
# 4. Tests (about 6 minutes); the notebook stops if any fails
for t in ['tests/test_v2xx.py', 'tests/test_v2xx_training.py', 'tests/test_v2xx_pipeline.py', 'tests/test_pipeline.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-3000:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
# 5. v2.03, official validation window: the 3 seeds train IN PARALLEL on one GPU (Siwat, 2026-10-25), then merge.
# Each seed is its own process ('--part'); it trains exactly as in a sequential run. Output lines are tagged [seed];
# each seed's full log is also saved to Drive. If Colab disconnects, Run all again: finished seeds are skipped.
import threading
def done(path, n=3):
    try:
        r = json.load(open(path)); return len(r['runs']) == n and 'summary' in r
    except Exception:
        return False
V203 = f'{OUT}/v2_03.json'
PARTS, WEIGHTS = f'{OUT}/v2_03_parts', f'{OUT}/v2_03_weights'
os.makedirs(PARTS, exist_ok=True)
def part_done(s):
    try:
        r = json.load(open(f'{PARTS}/seed{s}.json')); return bool(r.get('part')) and [x['seed'] for x in r['runs']] == [s]
    except Exception:
        return False
def pump(proc, tag, log_path):
    with open(log_path, 'w') as f:
        for line in proc.stdout:
            print(f'[{tag}] {line}', end=''); f.write(line); f.flush()
if done(V203):
    print('=== v2.03: already done')
else:
    procs = []
    for s in cfg['protocol']['seeds']:
        if part_done(s):
            print(f'=== seed {s}: already done'); continue
        print(f'=== seed {s}: starting'); sys.stdout.flush()
        pr = subprocess.Popen(['python', '-u', 'scripts/train_v2xx.py', '--config', 'configs/v2_03.json', '--part', '--seeds', str(s),
                               '--out', f'{PARTS}/seed{s}.json', '--save-weights', WEIGHTS],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        th = threading.Thread(target=pump, args=(pr, s, f'{PARTS}/seed{s}.log'), daemon=True); th.start()
        procs.append((s, pr, th))
    for s, pr, th in procs:
        pr.wait(); th.join()
    failed = [s for s, pr, th in procs if pr.returncode != 0]
    assert not failed, f'stop: seed(s) {failed} failed; see the lines tagged with that seed above, or {PARTS}/seed<seed>.log'
    subprocess.run(['python', 'scripts/merge_v2xx.py', '--config', 'configs/v2_03.json', '--parts', PARTS, '--out', V203], check=True)

In [ ]:
# 6. Summary beside v2.01 (L4 run, 2026-10-20), v2 and the floors
V201 = {'h1': 0.338, 'h5': 0.366, 'h10': 0.368}; V2 = {'h1': 0.387, 'h5': 0.396, 'h10': 0.378}
BAR = {'h1': 0.393, 'h5': 0.400, 'h10': 0.407}
r = json.load(open(V203)); s, fl = r['summary']['model'], r['floors']; ens = r.get('seed_ensemble', {}).get('val', {})
print(f"{'':6}{'v2.03 F1':>16}{'ensemble':>10}{'v2.01':>7}{'v2':>7}{'bar':>7}{'continues':>11}{'reverses':>10}{'R2_OS':>10}{'RMSE baht':>11}{'persist.':>10}")
for h in ('h1', 'h5', 'h10'):
    m = s[h]
    print(f"{h:6}{m['macro_f1']['mean']:>9.3f} ± {m['macro_f1']['std']:.3f}{ens.get(h, {}).get('macro_f1', float('nan')):>10.3f}"
          f"{V201[h]:>7.3f}{V2[h]:>7.3f}{BAR[h]:>7.3f}{fl[h]['trend_continues']['macro_f1']:>11.3f}{fl[h]['trend_reverses']['macro_f1']:>10.3f}"
          f"{m['r2_os_vs_zero']['mean']:>+10.4f}{m['price_rmse_baht']['mean']:>11.3f}{fl[h]['persistence_price_baht']['rmse']:>10.3f}")
for x in r['runs']:
    sw = [v for k, v in x.items() if k.startswith('scale_weights')]
    print('seed', x['seed'], 'epochs', x['epochs_run'], 'kept', x['best_epoch'], 'lowest CE epoch', x.get('ce_min_epoch'),
          'best F1 epoch overall', x.get('best_f1_epoch_overall'), 'params', x['params']['total'],
          'GPU peak GB', x['history'][-1].get('gpu_peak_gb'), 'scale weights', sw[0]['overall'] if sw else '-')

In [ ]:
# 7. Learning curves (+ CE, call agreement) and the call-share figure, saved beside the results on Drive
from IPython.display import Image, display
sys.path.insert(0, 'scripts')
import importlib, plot_v2xx_training as PT
importlib.reload(PT)
for png in (PT.plot(Path(V203)), PT.plot_calls(Path(V203))):
    if png:
        print('saved', png); display(Image(filename=str(png)))